In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os

spark = SparkSession.builder \
    .appName("CryptoPipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

print(f"Spark 버전: {spark.version}")

26/03/23 21:18:09 WARN Utils: Your hostname, DESKTOP-2NP0SM4 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/23 21:18:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/woo2910/.local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/woo2910/.ivy2/cache
The jars for the packages stored in: /home/woo2910/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2569f04-e198-40a5-b755-5110d791d34c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 622ms :: artifacts dl 34ms
	

Spark 버전: 3.5.3


In [2]:
schema = StructType([
    StructField("symbol", StringType()),
    StructField("price", DoubleType()),
    StructField("timestamp", LongType())
])

df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "raw-prices") \
    .load()

parsed_df = df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*") \
 .withColumn("timestamp", to_timestamp(col("timestamp") / 1000))

ma_df = parsed_df \
    .filter(col("price") > 0) \
    .filter(col("price") < 1000000) \
    .dropDuplicates(["symbol", "timestamp"]) \
    .groupBy(
        window(col("timestamp"), "1 minute"),
        col("symbol")
    ) \
    .agg(
        avg("price").alias("moving_avg"),
        min("price").alias("min_price"),
        max("price").alias("max_price")
    )

print("Spark Streaming 설정 완료!")

Spark Streaming 설정 완료!


In [3]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="crypto",
    user="postgres",
    password="password"
)

cursor = conn.cursor()
print("TimescaleDB 연결 성공!")

TimescaleDB 연결 성공!


In [4]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS crypto_prices (
        time        TIMESTAMPTZ NOT NULL,
        symbol      TEXT NOT NULL,
        moving_avg  DOUBLE PRECISION,
        min_price   DOUBLE PRECISION,
        max_price   DOUBLE PRECISION,
        source      TEXT DEFAULT 'realtime'
    );
""")

cursor.execute("""
    SELECT create_hypertable('crypto_prices', 'time', if_not_exists => TRUE);
""")

conn.commit()
print("테이블 생성 완료!")

테이블 생성 완료!


In [ ]:
def save_to_db(batch_df, batch_id):
    rows = batch_df.collect()
    for row in rows:
        cursor.execute("""
            INSERT INTO crypto_prices (time, symbol, moving_avg, min_price, max_price, source)
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            row['window']['end'],
            row['symbol'],
            row['moving_avg'],
            row['min_price'],
            row['max_price'],
            'realtime'
        ))
    conn.commit()
    print(f"배치 {batch_id} 저장 완료! ({len(rows)}개)")

os.makedirs("/tmp/checkpoint", exist_ok=True)

query = ma_df \
    .writeStream \
    .outputMode("update") \
    .foreachBatch(save_to_db) \
    .option("checkpointLocation", "/tmp/checkpoint") \
    .start()

print("스트리밍 시작!")
query.awaitTermination()

26/03/23 21:18:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


스트리밍 시작!


26/03/23 21:18:16 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/23 21:18:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/03/23 21:18:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/03/23 21:18:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/03/23 21:18:16 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
26/03/23 21:18:17 WARN NetworkClient: [AdminClient clientId=adminclient-1] Connection t

배치 0 저장 완료! (0개)


배치 1 저장 완료! (2개)


...

배치 378 저장 완료! (5개)


배치 379 저장 완료! (5개)


배치 380 저장 완료! (5개)


배치 381 저장 완료! (10개)


[Stage 1148:===================>                                (74 + 12) / 200]